# Module 4 — Demo lab (simulation and two-group tests)

**Not graded** · Four situations for [Project 4](../projects/project-4) · [Exercise lab](module-4-lab-exercise).

**Prerequisites:** [Module 4 Read 1](../lectures/module-4-read-01), [Read 2](../lectures/module-4-read-02).

| Problem | Situation |
|---------|-----------|
| 1 | Numerical outcome — Welch two-sample t-test |
| 2 | Binary outcome — two-proportion z-test |
| 3 | Validate under null (Type I error rate check) |
| 4 | RCT vs observational: see confounding in action |


## Code anatomy legend (demo notebooks only)

In **demo** notebooks we color-code **how to read R** ([best practices](../docs/best-practices/r-language-notes-and-demo-code.md)):

| Color | Meaning in code |
|-------|-----------------|
| <span style="color:#2563eb">Blue</span> | R functions and syntax (`mean`, `<-`, `()`, `~`) |
| <span style="color:#059669">Green</span> | Names **you** created (variables, data frames) |
| <span style="color:#dc2626">Red</span> | Values to **change** for your question or data |
| <span style="color:#7c3aed">Purple</span> | Important **output** to read carefully |

**Student exercise notebooks use plain code** — no instructor colors.

*Note:* Colab may highlight R syntax in its own colors. Our **anatomy colors** appear in markdown tables and examples below — they are a reading guide, not Colab's editor theme.


------------------------------------------------------------------------

## Demo — Four simulation + test situations

*Legacy Lab 11*

Same four scenarios as [Synthesis Section E](../quizzes/quiz-m4-synthesis).


## Setup

```{r}
library(ggplot2)
library(dplyr)
# install.packages(c("ggplot2", "dplyr"))  # if needed
```

**Quick guide:** Four simulation + two-group test situations for [Project 4](../projects/project-4):
numerical outcome (Welch t) · binary outcome (two-proportion z) · validate against known truth · RCT interpretation.


### Problem 1 — Numerical outcome: simulate and run Welch t-test

**Scenario:** Does a new sleep program increase nightly sleep hours compared to a control group?

- Treatment group ($n_1 = 30$): true mean 7.5 hrs, SD 1.0
- Control group ($n_2 = 30$): true mean 6.8 hrs, SD 1.0

**Tasks:** simulate · boxplot by group · Welch t-test · interpret output.


### Code anatomy — Problem 1

| Part | Role |
|------|------|
| `<span style="color:#2563eb">set.seed(42)</span>` | Fix random state for reproducibility |
| `<span style="color:#2563eb">rnorm(n, mean, sd)</span>` | Simulate normal outcome |
| `<span style="color:#2563eb">t.test(outcome ~ group)</span>` | Welch two-sample t-test |
| `<span style="color:#dc2626">data.frame(...)</span>` | Build tidy dataset for plotting |


In [ ]:
set.seed(42)

n1 <- 30
n2 <- 30

sleep <- c(rnorm(n1, mean = 7.5, sd = 1.0),
           rnorm(n2, mean = 6.8, sd = 1.0))
group <- c(rep("Treatment", n1), rep("Control", n2))

df <- data.frame(sleep = sleep, group = group)

# Descriptive summaries
df %>%
  group_by(group) %>%
  summarize(n = n(), mean = round(mean(sleep), 2), sd = round(sd(sleep), 2))

# Boxplot
ggplot(df, aes(x = group, y = sleep, fill = group)) +
  geom_boxplot(alpha = 0.7) +
  labs(title = "Nightly sleep by group (simulated)",
       x = "Group", y = "Hours of sleep") +
  theme(legend.position = "none")

# Welch two-sample t-test
t.test(sleep ~ group, data = df)


**Read output:** <span style="color:#7c3aed">p.value</span> — compare to $\alpha=0.05$. <span style="color:#7c3aed">conf.int</span> — 95% CI for population mean difference (Treatment $-$ Control). True difference is $7.5 - 6.8 = 0.7$ hrs — does the CI capture it?


### Problem 2 — Binary outcome: simulate and run two-proportion z-test

**Scenario:** Does a new drug increase recovery rates compared to a placebo?

- Treatment group ($n_1 = 80$): true recovery probability $p_1 = 0.65$
- Control group ($n_2 = 80$): true $p_2 = 0.45$

**Tasks:** simulate · bar chart · two-proportion z-test · interpret.


In [ ]:
set.seed(42)

n1 <- 80
n2 <- 80
p1 <- 0.65
p2 <- 0.45

recovered <- c(rbinom(n1, 1, p1), rbinom(n2, 1, p2))
group <- c(rep("Treatment", n1), rep("Control", n2))

df2 <- data.frame(recovered = factor(recovered, labels = c("No", "Yes")),
                  group = group)

# Descriptive proportions
df2 %>%
  group_by(group) %>%
  summarize(n = n(), recoveries = sum(recovered == "Yes"),
            prop = round(mean(recovered == "Yes"), 3))

# 100% stacked bar
ggplot(df2, aes(x = group, fill = recovered)) +
  geom_bar(position = "fill") +
  labs(title = "Recovery by group (simulated)",
       x = "Group", y = "Proportion", fill = "Recovered?")

# Two-proportion z-test
x1 <- sum(recovered[group == "Treatment"])
x2 <- sum(recovered[group == "Control"])
prop.test(c(x1, x2), c(n1, n2))


### Problem 3 — Validate against known truth

**Scenario:** Run 1000 simulated Welch t-tests under the null ($\mu_1 = \mu_2 = 5$, same SD).

**Goal:** verify that the false-positive rate (Type I error) is close to $\alpha = 0.05$.


In [ ]:
set.seed(123)

n_sim <- 1000
n_per_group <- 30
p_values <- numeric(n_sim)

for (i in seq_len(n_sim)) {
  g1 <- rnorm(n_per_group, mean = 5, sd = 2)
  g2 <- rnorm(n_per_group, mean = 5, sd = 2)  # same mean (null is TRUE)
  p_values[i] <- t.test(g1, g2)$p.value
}

# Type I error rate — should be near 0.05
mean(p_values < 0.05)

# Distribution of p-values under H0 — should be roughly uniform [0, 1]
ggplot(data.frame(p = p_values), aes(x = p)) +
  geom_histogram(binwidth = 0.05, boundary = 0) +
  labs(title = "P-value distribution under H0 (1000 simulations)",
       x = "p-value", y = "Count")


**Interpret:** If the simulation is correct, the false-positive rate should be near **0.05** and the p-value histogram should look approximately **uniform** — both are diagnostic tools for validating that your simulation code matches the stated null hypothesis.


### Problem 4 — RCT simulation with known confounder

**Scenario:** In an RCT, random assignment distributes a measured confounder (age group) across groups.

Compare a confounded observational assignment to an RCT assignment; see how confounding distorts the observed group difference.


In [ ]:
set.seed(42)

N <- 200

# True treatment effect: +3 units
# Confounder: older adults (age_group = "old") have higher baseline outcome AND are more likely to receive treatment (in observational setting)

age_group <- sample(c("young", "old"), N, replace = TRUE)
baseline <- ifelse(age_group == "old", 8, 5)  # confounder affects RV

# ---- Observational: older people more likely treated ----
prob_treat_obs <- ifelse(age_group == "old", 0.7, 0.3)
treat_obs <- rbinom(N, 1, prob_treat_obs)

outcome_obs <- baseline + 3 * treat_obs + rnorm(N, 0, 1)
df_obs <- data.frame(outcome = outcome_obs,
                     group = ifelse(treat_obs == 1, "Treatment", "Control"),
                     age_group = age_group)

cat("Observational naive t-test (confounded):\n")
print(t.test(outcome ~ group, data = df_obs)$estimate)

# ---- RCT: random assignment ----
treat_rct <- sample(c(0, 1), N, replace = TRUE)
outcome_rct <- baseline + 3 * treat_rct + rnorm(N, 0, 1)
df_rct <- data.frame(outcome = outcome_rct,
                     group = ifelse(treat_rct == 1, "Treatment", "Control"),
                     age_group = age_group)

cat("\nRCT t-test (unconfounded):\n")
print(t.test(outcome ~ group, data = df_rct)$estimate)
cat("True treatment effect: 3 units\n")


**Interpret:** The observational estimate is inflated by the confounder (older people are both more likely to be treated AND have higher baseline outcomes). The RCT estimate is closer to the **true treatment effect of 3 units** because random assignment breaks the age → treatment path.


---

### Optional — stratified analysis

See [Lab 11](../lab-11) for subclassification by a measured confounder using `group_by(age_group, treatment)`.
